In [23]:

"""
LEVEL 1 IMPLEMENTATION -- Pauli X Flip
"""

!pip install qiskit

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def make_message(name):
  m = QuantumCircuit(1)
  if name == '1':
    m.x(0)
  if name == '+':
    m.h(0)
  if name == '-':
    m.x(0)
    m.h(0)
  if name == '+i':
    m.h(0)
    m.s(0)
  if name == '-i':
    m.h(0)
    m.sdg(0)
  return m

# for debugging
def show_state(qc):
  # prints qubit bit string and their weights
  sv = Statevector(qc)
  for label, amp in sv.to_dict().items():
    print(label[::-1], np.round(amp, 3))

qc = QuantumCircuit(5) # q0, q1, q2 -- data; q3, q4 -- ancilla

# qc.h(0) # test state |+> on q0; 50/50 between 0 and 1 with weight of 1/rad(2)

qc.cx(0, 1) # CNOT q0 to q1
qc.cx(0, 2) # CNOT q0 to q2

show_state(qc)




00000 (1+0j)


In [22]:
qc = QuantumCircuit(5) # q0, q1, q2 -- data; q3, q4 -- ancilla

qc.h(0) # test state |+> on q0; 50/50 between 0 and 1 with weight of rad(2)
qc.cx(0, 1)
qc.cx(0, 2)

print("after encoding\n~~~~~~~~~~~~")
show_state(qc)
print()

qc.x(1) # Pauli X flip of the q1 qubit
print("after error\n~~~~~~~~~~~~")
show_state(qc)
print()

qc.cx(0, 3)
qc.cx(1, 3)
qc.cx(1, 4)
qc.cx(2, 4)
print("after ancilla\n~~~~~~~~~~~~")
show_state(qc)
print()

# Checking if 0 needs flipping
qc.x(4)
qc.ccx(3, 4, 0) # if 0 is wrong, 10 -- 11, Toffoli fires, 0 flipped
qc.x(4) # reset for next check

# Checking if 1 needs flipping
qc.ccx(3, 4, 1) # if 1 is wrong, 11 -- Toffoli fires for 1

# Checking if 2 needs flipping
qc.x(3)
qc.ccx(3, 4, 2) # if 2 is wrong, 01 -- 11, Toffoli fires, 2 gets flipped
qc.x(3) # reset
print("after correction\n~~~~~~~~~~~~")
show_state(qc)
print()

qc.cx(0, 2) # decodes message
qc.cx(0, 1)
print("after decoding\n~~~~~~~~~~~~")
show_state(qc)




after encoding
~~~~~~~~~~~~
00000 (0.707+0j)
11100 (0.707+0j)

after error
~~~~~~~~~~~~
01000 (0.707+0j)
10100 (0.707+0j)

after ancilla
~~~~~~~~~~~~
01011 (0.707+0j)
10111 (0.707+0j)

after correction
~~~~~~~~~~~~
00011 (0.707+0j)
11111 (0.707+0j)

after decoding
~~~~~~~~~~~~
00011 (0.707+0j)
10011 (0.707+0j)


In [24]:
for name in ['0','1', '+', '-', '+i', '-i']:
  for mine in [None, 0, 1, 2]:
    qc = QuantumCircuit(5)
    qc.compose(make_message(name), [0], inplace=True) # message is put on q0

    print("===== message", name, "| mine on", mine, "=====")
    print("Original:")
    show_state(qc)
    print()

    qc.cx(0, 1)
    qc.cx(0, 2)
    if mine is not None:
      qc.x(mine)
    print("With Error:")
    show_state(qc)
    print()

    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(1, 4)
    qc.cx(2, 4)

    qc.x(4)
    qc.ccx(3, 4, 0)
    qc.x(4)
    qc.ccx(3, 4, 1)
    qc.x(3)
    qc.ccx(3, 4, 2)
    qc.x(3)

    qc.cx(0, 2)
    qc.cx(0, 1)
    print("Fixed:")
    show_state(qc)
    print()

===== message 1 | mine on None =====
Original:
10000 (1+0j)

With Error:
11100 (1+0j)

Fixed:
10000 (1+0j)

===== message 1 | mine on 0 =====
Original:
10000 (1+0j)

With Error:
01100 (1+0j)

Fixed:
10010 (1+0j)

===== message 1 | mine on 1 =====
Original:
10000 (1+0j)

With Error:
10100 (1+0j)

Fixed:
10011 (1+0j)

===== message 1 | mine on 2 =====
Original:
10000 (1+0j)

With Error:
11000 (1+0j)

Fixed:
10001 (1+0j)

===== message + | mine on None =====
Original:
00000 (0.707+0j)
10000 (0.707+0j)

With Error:
00000 (0.707+0j)
11100 (0.707+0j)

Fixed:
00000 (0.707+0j)
10000 (0.707+0j)

===== message + | mine on 0 =====
Original:
00000 (0.707+0j)
10000 (0.707+0j)

With Error:
10000 (0.707+0j)
01100 (0.707+0j)

Fixed:
00010 (0.707+0j)
10010 (0.707+0j)

===== message + | mine on 1 =====
Original:
00000 (0.707+0j)
10000 (0.707+0j)

With Error:
01000 (0.707+0j)
10100 (0.707+0j)

Fixed:
00011 (0.707+0j)
10011 (0.707+0j)

===== message + | mine on 2 =====
Original:
00000 (0.707+0j)
10000 (0.

In [25]:
"""
LEVEL 2 IMPLEMENTATION -- Pauli Z Flip
"""

def show_state(qc):
  sv = Statevector(qc)
  for label, amp in sv.to_dict().items():
    if abs(amp) > 1e-6: # skips lines with 0 weight
      print(label[::-1], np.round(amp, 3))

for name in ['0', '1', '+', '-', '+i', '-i']:
  for mine in [None, 0, 1, 2]:
    qc = QuantumCircuit(5)
    qc.compose(make_message(name), [0], inplace=True)

    print("===== message", name, "| mine on", mine, "=====")
    print("Original:")
    show_state(qc)
    print()

    qc.cx(0, 1)
    qc.cx(0, 2)

    # First H gate Application
    qc.h(0)
    qc.h(1)
    qc.h(2)

    if mine is not None:
      qc.z(mine)

    # Second H gate Application -- Changed to X
    qc.h(0)
    qc.h(1)
    qc.h(2)

    print("With Error:")
    show_state(qc)
    print()

    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(1, 4)
    qc.cx(2, 4)

    qc.x(4)
    qc.ccx(3, 4, 0)
    qc.x(4)

    qc.ccx(3, 4, 1)
    qc.x(3)
    qc.ccx(3, 4, 2)
    qc.x(3)

    qc.cx(0, 2)
    qc.cx(0, 1)
    print("Fixed:")
    show_state(qc)
    print()


===== message 1 | mine on None =====
Original:
10000 (1+0j)

With Error:
11100 (1+0j)

Fixed:
10000 (1+0j)

===== message 1 | mine on 0 =====
Original:
10000 (1+0j)

With Error:
01100 (1+0j)

Fixed:
10010 (1+0j)

===== message 1 | mine on 1 =====
Original:
10000 (1+0j)

With Error:
10100 (1+0j)

Fixed:
10011 (1+0j)

===== message 1 | mine on 2 =====
Original:
10000 (1+0j)

With Error:
11000 (1+0j)

Fixed:
10001 (1+0j)

===== message + | mine on None =====
Original:
00000 (0.707+0j)
10000 (0.707+0j)

With Error:
00000 (0.707+0j)
11100 (0.707+0j)

Fixed:
00000 (0.707+0j)
10000 (0.707+0j)

===== message + | mine on 0 =====
Original:
00000 (0.707+0j)
10000 (0.707+0j)

With Error:
10000 (0.707+0j)
01100 (0.707+0j)

Fixed:
00010 (0.707+0j)
10010 (0.707+0j)

===== message + | mine on 1 =====
Original:
00000 (0.707+0j)
10000 (0.707+0j)

With Error:
01000 (0.707+0j)
10100 (0.707+0j)

Fixed:
00011 (0.707+0j)
10011 (0.707+0j)

===== message + | mine on 2 =====
Original:
00000 (0.707+0j)
10000 (0.

In [27]:
"""
LEVEL 3 IMPLEMENTATION -- Pauli Y Flip
"""

for name in ['0', '1', '+', '-', '+i', '-i']:
  for mine in [None, 0, 1, 2]:
    qc = QuantumCircuit(5)
    qc.compose(make_message(name), [0], inplace=True)

    print("===== message", name, "| mine on", mine, "=====")
    print("Original:")
    show_state(qc)
    print()

    qc.cx(0, 1)
    qc.cx(0, 2)

    # S applied -- rotates phase frame s.t. Sdg post Y flip will align w/ X flips
    qc.s(0)
    qc.s(1)
    qc.s(2)

    if mine is not None:
      qc.y(mine)
    # Sdg undoes S applied earlier -- rot. s.t. Y flip acts like X flip
    qc.sdg(0)
    qc.sdg(1)
    qc.sdg(2)

    print("With Error:")
    show_state(qc)
    print()

    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(1, 4)
    qc.cx(2, 4)

    qc.x(4)
    qc.ccx(3, 4, 0)
    qc.x(4)

    qc.ccx(3, 4, 1)
    qc.x(3)
    qc.ccx(3, 4, 2)
    qc.x(3)

    qc.cx(0, 2)
    qc.cx(0, 1)
    print("Fixed:")
    show_state(qc)
    print()

===== message 0 | mine on None =====
Original:
00000 (1+0j)

With Error:
00000 (1+0j)

Fixed:
00000 (1+0j)

===== message 0 | mine on 0 =====
Original:
00000 (1+0j)

With Error:
10000 (1+0j)

Fixed:
00010 (1+0j)

===== message 0 | mine on 1 =====
Original:
00000 (1+0j)

With Error:
01000 (1+0j)

Fixed:
00011 (1+0j)

===== message 0 | mine on 2 =====
Original:
00000 (1+0j)

With Error:
00100 (1+0j)

Fixed:
00001 (1+0j)

===== message 1 | mine on None =====
Original:
10000 (1+0j)

With Error:
11100 (1+0j)

Fixed:
10000 (1+0j)

===== message 1 | mine on 0 =====
Original:
10000 (1+0j)

With Error:
01100 (1+0j)

Fixed:
10010 (1+0j)

===== message 1 | mine on 1 =====
Original:
10000 (1+0j)

With Error:
10100 (1+0j)

Fixed:
10011 (1+0j)

===== message 1 | mine on 2 =====
Original:
10000 (1+0j)

With Error:
11000 (1+0j)

Fixed:
10001 (1+0j)

===== message + | mine on None =====
Original:
00000 (0.707+0j)
10000 (0.707+0j)

With Error:
00000 (0.707+0j)
11100 (0.707+0j)

Fixed:
00000 (0.707+0j)
1

In [1]:
"""
IQM Hardware Installation and Implementation
"""
!pip install "iqm-client[qiskit]"


In [10]:
"""
IQM Hardware Connect
"""

import sys, types
from IPython.display import display
_shim = types.ModuleType("IPython.core.display_functions")
_shim.display = display
sys.modules["IPython.core.display_functions"] = _shim

from getpass import getpass
from qiskit import QuantumCircuit, transpile
from iqm.qiskit_iqm import  IQMProvider

token = getpass("Paste IQM Token: ")
provider = IQMProvider("https://resonance.iqm.tech", quantum_computer="garnet", token=token)

backend = provider.get_backend()
print(backend.name, backend.num_qubits, backend.operation_names)


Paste IQM Token: ··········
garnet 19 ['id', 'delay', 'measure', 'r', 'if_else', 'reset', 'cz']


In [11]:
"""
IQM Hardware -- Lvl 1 Circuit w/Measurements
"""

def hardware_circuit(error_qubit):
  qc = QuantumCircuit(5, 3) # 5 qubits, 3 classical for measurement results
  qc.h(0) # message
  qc.cx(0, 1); qc.cx(0, 2) # encode
  qc.barrier()

  if error_qubit is not None:
    qc.x(error_qubit) # X flip
  qc.barrier()

  qc.cx(0, 3); qc.cx(1, 3); qc.cx(1, 4); qc.cx(2, 4) # ancilla
  qc.barrier()

  qc.x(4); qc.ccx(3, 4, 0); qc.x(4) # 10 -- flip q0
  qc.ccx(3, 4, 1) # 11 -- flip q1
  qc.x(3); qc.ccx(3, 4, 2); qc.x(3) # 01 -- flip q2
  qc.barrier()

  qc.cx(0, 2); qc.cx(0, 1) # decode
  qc.h(0)
  qc.measure(0, 0); qc.measure(3, 1); qc.measure(4, 2) # measure
  return qc

print(hardware_circuit(1).draw())

     ┌───┐           ░       ░                      ░      ┌───┐               »
q_0: ┤ H ├──■────■───░───────░───■──────────────────░──────┤ X ├───────────────»
     └───┘┌─┴─┐  │   ░ ┌───┐ ░   │                  ░      └─┬─┘     ┌───┐     »
q_1: ─────┤ X ├──┼───░─┤ X ├─░───┼────■────■────────░────────┼───────┤ X ├─────»
          └───┘┌─┴─┐ ░ └───┘ ░   │    │    │        ░        │       └─┬─┘     »
q_2: ──────────┤ X ├─░───────░───┼────┼────┼────■───░────────┼─────────┼───────»
               └───┘ ░       ░ ┌─┴─┐┌─┴─┐  │    │   ░        │         │  ┌───┐»
q_3: ────────────────░───────░─┤ X ├┤ X ├──┼────┼───░────────■─────────■──┤ X ├»
                     ░       ░ └───┘└───┘┌─┴─┐┌─┴─┐ ░ ┌───┐  │  ┌───┐  │  └───┘»
q_4: ────────────────░───────░───────────┤ X ├┤ X ├─░─┤ X ├──■──┤ X ├──■───────»
                     ░       ░           └───┘└───┘ ░ └───┘     └───┘          »
c: 3/══════════════════════════════════════════════════════════════════════════»
                            

In [12]:
"""
IQM Hardware -- run
"""
import json

cases = [None, 0, 1, 2]
circuits = [hardware_circuit(e) for e in cases]
transpiled = transpile(circuits, backend=backend, optimization_level=3)   # rebuild in the chip's native gates
job = backend.run(transpiled, shots=1000)
print("job id:", job.job_id())   # save this in case Colab disconnects

result = job.result()
all_counts = {}
for i, e in enumerate(cases):
    counts = result.get_counts(i)
    all_counts[str(e)] = counts
    print("mine on", e, ":", counts)

# save for the GitHub repo
with open(f"iqm_results_{backend.name}.json", "w") as f:
    json.dump({"backend": backend.name, "job_id": str(job.job_id()), "counts": all_counts}, f, indent=2)

job id: 01a20346-83a5-75a6-98e7-17e494fc2f6d
mine on None : {'000': 649, '011': 30, '001': 169, '110': 18, '101': 30, '010': 38, '111': 24, '100': 42}
mine on 0 : {'000': 62, '010': 634, '011': 145, '001': 36, '111': 33, '110': 47, '100': 27, '101': 16}
mine on 1 : {'011': 54, '110': 582, '000': 30, '010': 83, '100': 41, '111': 150, '101': 39, '001': 21}
mine on 2 : {'100': 523, '101': 259, '110': 42, '000': 60, '111': 29, '001': 54, '011': 15, '010': 18}
